# 🏎️ Projet F1 — Partie 1 : Ingestion & Préparation des données
**Responsable : Lara**

Cette partie couvre :
- Initialisation de la session Spark
- Chargement des fichiers CSV avec schémas explicites
- Gestion des valeurs manquantes, doublons et types incorrects
- Documentation des choix

## 1. Initialisation de la session Spark

In [17]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] = "C:\\hadoop\\bin;" + os.environ["PATH"]
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, FloatType, DoubleType
)

spark = SparkSession.builder \
    .appName("F1_Projet_Groupe") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Session Spark initialisée")
print(f"   Version Spark : {spark.version}")

✅ Session Spark initialisée
   Version Spark : 3.5.3


## 2. Définition des schémas explicites

**Choix documenté :** On définit les schémas manuellement plutôt qu'avec `inferSchema=True`.
Cela permet d'éviter un double scan du fichier et de garantir des types corrects dès le chargement.

In [18]:
# ---------- drivers.csv ----------
schema_drivers = StructType([
    StructField("driverId",      IntegerType(), True),
    StructField("driverRef",     StringType(),  True),
    StructField("number",        StringType(),  True),  # peut contenir '\\N'
    StructField("code",          StringType(),  True),
    StructField("forename",      StringType(),  True),
    StructField("surname",       StringType(),  True),
    StructField("dob",           StringType(),  True),  # converti plus bas
    StructField("nationality",   StringType(),  True),
    StructField("url",           StringType(),  True),
])

# ---------- results.csv ----------
schema_results = StructType([
    StructField("resultId",       IntegerType(), True),
    StructField("raceId",         IntegerType(), True),
    StructField("driverId",       IntegerType(), True),
    StructField("constructorId",  IntegerType(), True),
    StructField("number",         StringType(),  True),
    StructField("grid",           IntegerType(), True),
    StructField("position",       StringType(),  True),  # peut contenir '\\N'
    StructField("positionText",   StringType(),  True),
    StructField("positionOrder",  IntegerType(), True),
    StructField("points",         FloatType(),   True),
    StructField("laps",           IntegerType(), True),
    StructField("time",           StringType(),  True),
    StructField("milliseconds",   StringType(),  True),
    StructField("fastestLap",     StringType(),  True),
    StructField("rank",           StringType(),  True),
    StructField("fastestLapTime", StringType(),  True),
    StructField("fastestLapSpeed",StringType(),  True),
    StructField("statusId",       IntegerType(), True),
])

# ---------- races.csv ----------
schema_races = StructType([
    StructField("raceId",    IntegerType(), True),
    StructField("year",      IntegerType(), True),
    StructField("round",     IntegerType(), True),
    StructField("circuitId", IntegerType(), True),
    StructField("name",      StringType(),  True),
    StructField("date",      StringType(),  True),
    StructField("time",      StringType(),  True),
    StructField("url",       StringType(),  True),
])

# ---------- constructors.csv ----------
schema_constructors = StructType([
    StructField("constructorId",  IntegerType(), True),
    StructField("constructorRef", StringType(),  True),
    StructField("name",           StringType(),  True),
    StructField("nationality",    StringType(),  True),
    StructField("url",            StringType(),  True),
])

# ---------- constructor_standings.csv ----------
schema_constructor_standings = StructType([
    StructField("constructorStandingsId", IntegerType(), True),
    StructField("raceId",                 IntegerType(), True),
    StructField("constructorId",          IntegerType(), True),
    StructField("points",                 FloatType(),   True),
    StructField("position",               IntegerType(), True),
    StructField("positionText",           StringType(),  True),
    StructField("wins",                   IntegerType(), True),
])

# ---------- driver_standings.csv ----------
schema_driver_standings = StructType([
    StructField("driverStandingsId", IntegerType(), True),
    StructField("raceId",            IntegerType(), True),
    StructField("driverId",          IntegerType(), True),
    StructField("points",            FloatType(),   True),
    StructField("position",          IntegerType(), True),
    StructField("positionText",      StringType(),  True),
    StructField("wins",              IntegerType(), True),
])

# ---------- circuits.csv ----------
schema_circuits = StructType([
    StructField("circuitId",  IntegerType(), True),
    StructField("circuitRef", StringType(),  True),
    StructField("name",       StringType(),  True),
    StructField("location",   StringType(),  True),
    StructField("country",    StringType(),  True),
    StructField("lat",        DoubleType(),  True),
    StructField("lng",        DoubleType(),  True),
    StructField("alt",        StringType(),  True),
    StructField("url",        StringType(),  True),
])


# ---------- qualifying.csv ----------
schema_qualifying = StructType([
    StructField("qualifyId",     IntegerType(), True),
    StructField("raceId",        IntegerType(), True),
    StructField("driverId",      IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("number",        IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("q1",            StringType(),  True),
    StructField("q2",            StringType(),  True),
    StructField("q3",            StringType(),  True),
])

print("✅ Schémas définis pour 8 tables")

✅ Schémas définis pour 7 tables


## 3. Chargement des fichiers CSV

**Choix documenté :** Le dataset Kaggle utilise `\\N` pour représenter les valeurs nulles (convention MySQL). On le spécifie dans `nullValue` pour que Spark les traite correctement comme `null`.

In [19]:
# ⚠️ Adapter ce chemin selon votre environnement
DATA_PATH = "../data/raw/"   # ex: "/content/f1/" sur Colab, ou "./data/" en local

def load_csv(filename, schema):
    """Charge un CSV F1 avec les options adaptées au dataset Kaggle."""
    return spark.read \
        .option("header", True) \
        .option("nullValue", "\\N") \
        .option("escape", '"') \
        .schema(schema) \
        .csv(DATA_PATH + filename)

df_drivers              = load_csv("drivers.csv",              schema_drivers)
df_results              = load_csv("results.csv",              schema_results)
df_races                = load_csv("races.csv",                schema_races)
df_constructors         = load_csv("constructors.csv",         schema_constructors)
df_constructor_standings= load_csv("constructor_standings.csv",schema_constructor_standings)
df_driver_standings     = load_csv("driver_standings.csv",     schema_driver_standings)
df_circuits             = load_csv("circuits.csv",             schema_circuits)

df_qualifying           = load_csv("qualifying.csv",          schema_qualifying)

print("✅ Tous les fichiers chargés")

✅ Tous les fichiers chargés


## 4. Exploration initiale — Aperçu des données

In [20]:
tables = {
    "drivers":               df_drivers,
    "results":               df_results,
    "races":                 df_races,
    "constructors":          df_constructors,
    "constructor_standings": df_constructor_standings,
    "driver_standings":      df_driver_standings,
    "circuits":              df_circuits,
    "qualifying":            df_qualifying,
}

print(f"{'Table':<30} {'Lignes':>10} {'Colonnes':>10}")
print("-" * 52)
for name, df in tables.items():
    print(f"{name:<30} {df.count():>10} {len(df.columns):>10}")

Table                              Lignes   Colonnes
----------------------------------------------------
drivers                               861          9
results                             26759         18
races                                1125          8
constructors                          212          5
constructor_standings               13391          7
driver_standings                    34863          7
circuits                               77          9


In [21]:
# Aperçu de chaque table
for name, df in tables.items():
    print(f"\n{'='*60}")
    print(f"📋 {name.upper()}")
    print(f"{'='*60}")
    df.printSchema()
    df.show(3, truncate=True)


📋 DRIVERS
root
 |-- driverId: integer (nullable = true)
 |-- driverRef: string (nullable = true)
 |-- number: string (nullable = true)
 |-- code: string (nullable = true)
 |-- forename: string (nullable = true)
 |-- surname: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- url: string (nullable = true)

+--------+---------+------+----+--------+--------+----------+-----------+--------------------+
|driverId|driverRef|number|code|forename| surname|       dob|nationality|                 url|
+--------+---------+------+----+--------+--------+----------+-----------+--------------------+
|       1| hamilton|    44| HAM|   Lewis|Hamilton|1985-01-07|    British|http://en.wikiped...|
|       2| heidfeld|  NULL| HEI|    Nick|Heidfeld|1977-05-10|     German|http://en.wikiped...|
|       3|  rosberg|     6| ROS|    Nico| Rosberg|1985-06-27|     German|http://en.wikiped...|
+--------+---------+------+----+--------+--------+----------+----

## 5. Gestion des valeurs manquantes

**Choix documentés :**
- `position` dans results : les DNF/DNS ont `null` → on les conserve avec la valeur `positionText` qui est plus informative
- `number` des pilotes : certains pilotes anciens n'ont pas de numéro permanent → on garde `null`
- `milliseconds` : utilisé pour les analyses de temps, les lignes sans cette valeur sont exclues uniquement dans les analyses de temps

In [22]:
def count_nulls(df, name):
    """Affiche le nombre de nulls par colonne."""
    print(f"\n🔍 Valeurs nulles — {name}")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])
    null_counts.show(truncate=False)

for name, df in tables.items():
    count_nulls(df, name)


🔍 Valeurs nulles — drivers
+--------+---------+------+----+--------+-------+---+-----------+---+
|driverId|driverRef|number|code|forename|surname|dob|nationality|url|
+--------+---------+------+----+--------+-------+---+-----------+---+
|0       |0        |802   |757 |0       |0      |0  |0          |0  |
+--------+---------+------+----+--------+-------+---+-----------+---+


🔍 Valeurs nulles — results
+--------+------+--------+-------------+------+----+--------+------------+-------------+------+----+-----+------------+----------+-----+--------------+---------------+--------+
|resultId|raceId|driverId|constructorId|number|grid|position|positionText|positionOrder|points|laps|time |milliseconds|fastestLap|rank |fastestLapTime|fastestLapSpeed|statusId|
+--------+------+--------+-------------+------+----+--------+------------+-------------+------+----+-----+------------+----------+-----+--------------+---------------+--------+
|0       |0     |0       |0            |6     |0   |10953   |0

In [23]:
# --- Nettoyage drivers ---
# On garde les nulls sur 'number' et 'code' (données historiques incomplètes, c'est normal)
# On supprime les lignes sans driverId ou surname (clé métier indispensable)
df_drivers_clean = df_drivers.dropna(subset=["driverId", "surname", "forename"])

# Conversion de la date de naissance en DateType
df_drivers_clean = df_drivers_clean.withColumn(
    "dob", F.to_date(F.col("dob"), "yyyy-MM-dd")
)

# Création du nom complet
df_drivers_clean = df_drivers_clean.withColumn(
    "fullName", F.concat_ws(" ", F.col("forename"), F.col("surname"))
)

print(f"✅ drivers : {df_drivers.count()} → {df_drivers_clean.count()} lignes après nettoyage")
df_drivers_clean.show(5)

✅ drivers : 861 → 861 lignes après nettoyage
+--------+----------+------+----+--------+----------+----------+-----------+--------------------+-----------------+
|driverId| driverRef|number|code|forename|   surname|       dob|nationality|                 url|         fullName|
+--------+----------+------+----+--------+----------+----------+-----------+--------------------+-----------------+
|       1|  hamilton|    44| HAM|   Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|   Lewis Hamilton|
|       2|  heidfeld|  NULL| HEI|    Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|    Nick Heidfeld|
|       3|   rosberg|     6| ROS|    Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|     Nico Rosberg|
|       4|    alonso|    14| ALO|Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|  Fernando Alonso|
|       5|kovalainen|  NULL| KOV|  Heikki|Kovalainen|1981-10-19|    Finnish|http://en.wikiped...|Heikki Kovalainen|
+--------+----------+------

In [24]:
# --- Nettoyage results ---
# 'position' null = le pilote n'a pas terminé (DNF, DSQ...) → on conserve
# On convertit 'position' en entier quand possible
df_results_clean = df_results.withColumn(
    "position_int",
    F.col("position").cast(IntegerType())
)

# Flag pour savoir si le pilote a terminé la course
df_results_clean = df_results_clean.withColumn(
    "finished",
    F.when(F.col("position_int").isNotNull(), True).otherwise(False)
)

# Conversion milliseconds
df_results_clean = df_results_clean.withColumn(
    "milliseconds", F.col("milliseconds").cast("long")
)

# Suppression des lignes sans clés métier essentielles
df_results_clean = df_results_clean.dropna(subset=["resultId", "raceId", "driverId"])

print(f"✅ results : {df_results.count()} → {df_results_clean.count()} lignes après nettoyage")

✅ results : 26759 → 26759 lignes après nettoyage


In [25]:
# --- Nettoyage races ---
df_races_clean = df_races.dropna(subset=["raceId", "year", "name"])

# Conversion date
df_races_clean = df_races_clean.withColumn(
    "date", F.to_date(F.col("date"), "yyyy-MM-dd")
)

print(f"✅ races : {df_races.count()} → {df_races_clean.count()} lignes après nettoyage")

# --- Nettoyage constructors ---
df_constructors_clean = df_constructors.dropna(subset=["constructorId", "name"])
print(f"✅ constructors : {df_constructors.count()} → {df_constructors_clean.count()} lignes après nettoyage")

# --- Nettoyage driver_standings ---
df_driver_standings_clean = df_driver_standings.dropna(subset=["driverStandingsId", "raceId", "driverId"])
print(f"✅ driver_standings : {df_driver_standings.count()} → {df_driver_standings_clean.count()} lignes après nettoyage")

# --- Nettoyage qualifying ---
df_qualifying_clean = df_qualifying.dropna(subset=["qualifyId", "raceId", "driverId"])
# Flag pole position
df_qualifying_clean = df_qualifying_clean.withColumn("pole_position", F.col("position") == 1)
# Flag Q3 atteinte (top 10)
df_qualifying_clean = df_qualifying_clean.withColumn("reached_q3", F.col("q3").isNotNull())
print(f"✅ qualifying : {df_qualifying.count()} → {df_qualifying_clean.count()} lignes après nettoyage")


✅ races : 1125 → 1125 lignes après nettoyage
✅ constructors : 212 → 212 lignes après nettoyage
✅ driver_standings : 34863 → 34863 lignes après nettoyage


## 6. Gestion des doublons

In [26]:
def check_and_remove_duplicates(df, name, key_cols):
    """Vérifie et supprime les doublons sur les colonnes clés."""
    total    = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dupes    = total - distinct
    print(f"🔁 {name:<30} | Total: {total:>7} | Doublons sur {key_cols}: {dupes}")
    return df.dropDuplicates(key_cols)

df_drivers_clean              = check_and_remove_duplicates(df_drivers_clean,              "drivers",              ["driverId"])
df_results_clean              = check_and_remove_duplicates(df_results_clean,              "results",              ["resultId"])
df_races_clean                = check_and_remove_duplicates(df_races_clean,                "races",                ["raceId"])
df_constructors_clean         = check_and_remove_duplicates(df_constructors_clean,         "constructors",         ["constructorId"])
df_driver_standings_clean     = check_and_remove_duplicates(df_driver_standings_clean,     "driver_standings",     ["driverStandingsId"])
df_constructor_standings_clean= check_and_remove_duplicates(df_constructor_standings,      "constructor_standings",["constructorStandingsId"])
df_qualifying_clean           = check_and_remove_duplicates(df_qualifying_clean,           "qualifying",           ["qualifyId"])

🔁 drivers                        | Total:     861 | Doublons sur ['driverId']: 0
🔁 results                        | Total:   26759 | Doublons sur ['resultId']: 0
🔁 races                          | Total:    1125 | Doublons sur ['raceId']: 0
🔁 constructors                   | Total:     212 | Doublons sur ['constructorId']: 0
🔁 driver_standings               | Total:   34863 | Doublons sur ['driverStandingsId']: 0
🔁 constructor_standings          | Total:   13391 | Doublons sur ['constructorStandingsId']: 0


## 7. Vérification des types finaux

In [27]:
print("📐 Schéma final — drivers")
df_drivers_clean.printSchema()

print("\n📐 Schéma final — results")
df_results_clean.printSchema()

print("\n📐 Schéma final — races")
df_races_clean.printSchema()

📐 Schéma final — drivers
root
 |-- driverId: integer (nullable = true)
 |-- driverRef: string (nullable = true)
 |-- number: string (nullable = true)
 |-- code: string (nullable = true)
 |-- forename: string (nullable = true)
 |-- surname: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- nationality: string (nullable = true)
 |-- url: string (nullable = true)
 |-- fullName: string (nullable = false)


📐 Schéma final — results
root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: string (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: string (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: float (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: long (nullable = true)
 |-- fastestLap: string (nu

## 8. Mise en cache pour les parties suivantes

**Choix documenté :** On met en cache les tables les plus utilisées (drivers, results, races) pour éviter de les relire depuis le disque à chaque transformation. Les tables volumineuses comme `results` bénéficient le plus de cette optimisation.

In [28]:
df_drivers_clean.cache()
df_results_clean.cache()
df_races_clean.cache()
df_constructors_clean.cache()
df_qualifying_clean.cache()

# Forcer le chargement en mémoire
_ = df_drivers_clean.count()
_ = df_results_clean.count()
_ = df_races_clean.count()

print("✅ Tables mises en cache")

✅ Tables mises en cache


## 9. Export des DataFrames nettoyés (pour les autres parties)

On sauvegarde les DataFrames nettoyés au format **Parquet** : plus efficace que CSV pour Spark (colonne compressée, typage conservé).

In [31]:
OUTPUT_PATH = "../data/processed/"

df_drivers_clean.toPandas().to_csv(OUTPUT_PATH + "drivers_clean.csv", index=False)
df_results_clean.toPandas().to_csv(OUTPUT_PATH + "results_clean.csv", index=False)
df_races_clean.toPandas().to_csv(OUTPUT_PATH + "races_clean.csv", index=False)
df_constructors_clean.toPandas().to_csv(OUTPUT_PATH + "constructors_clean.csv", index=False)
df_driver_standings_clean.toPandas().to_csv(OUTPUT_PATH + "driver_standings_clean.csv", index=False)
df_constructor_standings_clean.toPandas().to_csv(OUTPUT_PATH + "constructor_standings_clean.csv", index=False)

df_qualifying_clean.toPandas().to_csv(OUTPUT_PATH + "qualifying_clean.csv", index=False)

print("✅ DataFrames nettoyés exportés en CSV dans", OUTPUT_PATH)

✅ DataFrames nettoyés exportés en CSV dans ../data/processed/


## 10. Récapitulatif des choix de nettoyage

| Table | Traitement appliqué |
|---|---|
| **drivers** | Suppression si `driverId`/`surname`/`forename` null • `dob` → DateType • Création `fullName` |
| **results** | `position` → IntegerType + flag `finished` • `milliseconds` → LongType • Suppression si clé nulle |
| **races** | `date` → DateType • Suppression si `raceId`/`year`/`name` null |
| **constructors** | Suppression si `constructorId`/`name` null |
| **driver_standings** | Suppression si clés primaires nulles |
| **qualifying** | Suppression si clés primaires nulles • flag `pole_position` • flag `reached_q3` |
| **Toutes** | `nullValue='\\N'` géré au chargement • Déduplication sur clé primaire |

**Valeurs conservées intentionnellement :**
- `position` null dans `results` → correspond à un abandon (DNF/DNS), information métier utile
- `number` null dans `drivers` → numéros non attribués aux pilotes anciens
- `code` null dans `drivers` → abréviations manquantes pour l'ère ancienne